# Dynamic tools — a different toolbox per user role
**by Kassouar Fatima 🇩🇿 — Module 3 project**

The agent is built with **both** tools — `web_search` (Tavily) and `sql_query` (the bundled Chinook SQLite database) — but a `@wrap_model_call` middleware reads `user_role` from the context and, for `external` users, overrides the tool list down to `web_search` only: ask the SQL question as an external user and the model literally cannot see the database tool.

- **Run:** double-click `run_3_4.cmd` (opens all three `3_4` notebooks) → *Run → Run All Cells*.
- **Needs:** `GOOGLE_API_KEY` **and** `TAVILY_API_KEY` — the `.env` next to this notebook.
- **Model:** `gemini-3.5-flash-lite` — its own 500/day bucket, free while `gemini-3.1-flash-lite` is spent. **Model fix:** the course used `gpt-5-nano` → Gemini here, no extra key.

**Idea:** tools are a field of the request like any other — permissions are enforced per call, at dispatch time, not baked into the agent.

**What I learned:**
- `request.override(tools=[...])` shrinks the toolbox for this call only — the agent still holds both tools.
- The role travels in context: `context_schema=UserRole` + `context={"user_role": "external"}` at invoke time.
- With `external`, the "artists in the database" question comes from the web instead — that mismatch *is* the demo (`internal` lights up `sql_query`).


## Setup

In [1]:
import importlib.util as _ilu
import subprocess as _sp
import sys as _sys

# Install only what is missing. Uses pip when available, falls back to uv
# (the local venv on this machine is created by uv and has no pip).
REQUIRED = {
    "langchain": "langchain",
    "langgraph": "langgraph",
    "langchain_google_genai": "langchain-google-genai",
    "langchain_community": "langchain-community",
    "dotenv": "python-dotenv",
    "tavily": "tavily-python",
}
missing = [pkg for imp, pkg in REQUIRED.items() if _ilu.find_spec(imp) is None]

if missing:
    print("Installing:", ", ".join(missing))
    if _sp.run([_sys.executable, "-m", "pip", "install", "-q", *missing]).returncode != 0:
        _sp.run(["uv", "pip", "install", *missing])
else:
    print("All packages already installed.")

All packages already installed.


In [2]:
import os
from getpass import getpass
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))  # local run: finds the .env above the notebook folder

# Colab run: read the key from the Secrets panel (key icon in the left sidebar)
try:
    from google.colab import userdata
    try:
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except Exception:
        pass
except ImportError:
    pass

def is_real(value):
    # empty values and template placeholders ("your_..." ) count as missing
    return bool(value) and not value.startswith("your_")

# Accept both names: the local project keeps the key as GEMINI_API_KEY
for name in ("GOOGLE_API_KEY", "GEMINI_API_KEY"):
    if is_real(os.getenv(name)):
        os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"] = os.getenv(name)
        break

# Still missing? Paste it here (hidden input).
if not is_real(os.getenv("GOOGLE_API_KEY")):
    try:
        value = getpass("Gemini API key is missing - paste it (Enter to skip): ").strip()
    except Exception:
        value = ""
    if value:
        os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"] = value

print("GOOGLE_API_KEY:", "set" if is_real(os.getenv("GOOGLE_API_KEY")) else "MISSING")

# Tracing with a placeholder LangSmith key only produces 401/403 warnings: switch it off
if not is_real(os.getenv("LANGSMITH_API_KEY")):
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing: off (no valid LANGSMITH_API_KEY)")

GOOGLE_API_KEY: set
LangSmith tracing: off (no valid LANGSMITH_API_KEY)


In [3]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase

tavily_client = TavilyClient()

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")


@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

C:\Users\Kassouar Fatima\AppData\Local\Temp\ipykernel_26220\4208644247.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [4]:
from dataclasses import dataclass

@dataclass
class UserRole:
    user_role: str = "external"

In [5]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Dynamically call tools based on the runtime context"""

    user_role = request.runtime.context.user_role
    
    if user_role == "internal":
        pass # internal users get access to all tools
    else:
        tools = [web_search] # external users only get access to web search
        request = request.override(tools=tools) 

    return handler(request)

In [6]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call],
    context_schema=UserRole
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [7]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "external"}
)

print(response["messages"][-1].content)

[{'type': 'text', 'text': "It looks like you didn't mention *which* database you are referring to! Could you please clarify what database or platform you are asking about (for example, Spotify, Apple Music, Discogs, a specific video game, or a company database)?", 'extras': {'signature': 'EmAKXgFpFH0T3/S89PDD0WEpDFCf4esib1z95YvIwT4h/PG9H5sqop9g5eN6VS+SM1b5RXq90Q8vjwlPmGzOieCVk08Te0qLthAM8cSKeIPfgYc7vY4Cs48vyyM5Dy0P7CU='}}]


## Your task

> - Make one prompt dynamic. Anything: language, tone, level of detail. Get it switching on something real.
> - Hide a tool from one kind of user. This is the pattern behind every permission you will ever implement.
> - Then try the model switch, and put the token counts side by side. That is the difference between an agent that works and an agent that can afford to run.

### My response — by Kassouar Fatima 🇩🇿 — Module 3 project

This notebook does **task 2**; tasks 1 and 3 live in the two sibling notebooks
(`3_4_Dynamic_prompts_kassouar`, `3_4_Dynamic_models_kassouar`).

**Hiding the tool.** The agent is built with **both** tools registered — `tools=[web_search, sql_query]`
— and the `@wrap_model_call` middleware narrows them per call:

```python
user_role = request.runtime.context.user_role
if user_role == "internal":
    pass                                    # internal users get access to all tools
else:
    tools = [web_search]                    # external users only get access to web search
    request = request.override(tools=tools)
return handler(request)
```

The permission is enforced in **code**, at dispatch time — not by telling the model to behave.

**The proof** (the notebook runs the external case with `context={"user_role": "external"}`):

> *How many artists are in the database?*
>
> **"It looks like you didn't mention *which* database you are referring to! Could you please
> clarify what database or platform you are asking about (for example, Spotify, Apple Music,
> Discogs, a specific video game, or a company database)?"**

The model did not run a single query. With `sql_query` removed from *its* view of the toolbox it
had no way to reach Chinook, so it fell back to asking me which database I meant — it never even
knew the SQL tool existed. `UserRole` defaults to `"external"`, so the safe answer is the default:
you have to be explicitly internal to get the database.

**Why this is the pattern behind every permission.** Nothing was unregistered, no second agent was
built, no prompt said *"never use SQL"* — one `if`, one `request.override(tools=...)`. Approve,
read-only, internal, admin — they are all this shape: read a role from context, override the
request, hand it to the handler. The tool list becomes a property of *who is asking*, which is
exactly what a permission is.

## Summary — main functions & parameters

| Function / parameter | What it does |
|---|---|
| `@tool web_search(query)` | Tavily search — the only tool an `external` user keeps |
| `@tool sql_query(query)` | Chinook SQLite — hidden from `external` (and `langchain-community` warns it is being sunset) |
| `@dataclass UserRole(user_role="external")` | the per-request role; **defaults to `external`**, so the safe answer is the default |
| `@wrap_model_call dynamic_tool_call(request, handler)` | runs before every model call and may narrow the toolbox |
| `request.runtime.context.user_role`", | where the role is read from |
| `request.override(tools=[web_search])` | shrink the toolbox **for this call only** — `internal` hits `pass` and keeps both |
| `create_agent(..., tools=[web_search, sql_query], context_schema=UserRole)` | both tools always registered; the middleware decides who sees what |
| `agent.invoke(..., context={"user_role": "external"})` | a plain dict works too — it is coerced to the context schema |

---
*by Kassouar Fatima 🇩🇿 — Module 3 project*